In [1]:
names = open('names.txt', 'r').read().splitlines()
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

### Dataset builder

In [2]:
vocab = sorted(list(set(''.join(names))))
vocab.append('.')

char_to_token = {c : i for i, c in enumerate(vocab)}
token_to_char = {i : c for c, i in char_to_token.items()}

In [3]:
import torch

xs = []
ys = []
for name in names:
    chars = ['.'] + list(name) + ['.']
    for c1, c2 in zip(chars, chars[1:]):
        xs.append(char_to_token[c1])
        ys.append(char_to_token[c2])

x_train = torch.tensor(xs)
y_train = torch.tensor(ys)

### Training


In [12]:
from torch.nn import functional as F
g = torch.Generator().manual_seed(100)

W = torch.randn((len(vocab), len(vocab)), requires_grad=True, generator=g)
alpha = 20

def forward(x):
    xenc = F.one_hot(x, num_classes=27).float()
    logits = xenc.matmul(W) # log_counts
    return logits

def nll_loss(logits, y):
    nll = -F.log_softmax(logits, dim=1)
    nll = nll[range(len(y)), y]
    nll = nll.mean()
    return nll

for i in range(1000):
    logits = forward(x_train)
    nll = nll_loss(logits, y_train)
    loss = nll
    
    W.grad = None
    loss.backward()
    if i % 100 == 0:
        print(loss.item())
    
    W.data += -alpha * W.grad
        

3.5823442935943604
2.5053417682647705
2.477930784225464
2.4688847064971924
2.4646668434143066
2.4622561931610107
2.4607040882110596
2.459623336791992
2.458829641342163
2.4582231044769287


### Inference

In [13]:
def generate_names(num_names, char, W):
    names = []
    for i in range(num_names):
        out = []
        next_token = char_to_token[char]
        
        while True:
            xenc = F.one_hot(torch.tensor([next_token]), 
                             num_classes=len(vocab)).float()
            logits = xenc.matmul(W)
            probs = F.softmax(logits, dim=1)
            
            next_token = torch.multinomial(probs, num_samples=1, 
                                          replacement=True, 
                                          generator=g).item()
            out.append(token_to_char[next_token])
            
            if next_token == char_to_token['.']: break
        name = "".join(out)
        names.append(name)
    
    return names

names_ = generate_names(5, ".", W)
names_

['joerrieriynol.', 'anyeviderndeyandei.', 'kr.', 'morae.', 'a.']

Performing the forward pass manually VS. using PyTorch

In [14]:
from torch.nn import functional as F 

logits_ = F.one_hot(x_train, num_classes=27).float() @ W # log_counts
print(f'log_counts: \n{logits_[:3]}\n')

log_counts: 
tensor([[ 1.7404e+00,  5.2324e-01,  6.8940e-01,  7.8108e-01,  6.8224e-01,
         -6.1908e-01, -1.4600e-01,  1.2144e-01, -2.7005e-01,  1.1410e+00,
          1.3427e+00,  7.0868e-01,  1.1878e+00,  3.9251e-01, -6.7588e-01,
         -4.0781e-01, -2.1344e+00,  7.5043e-01,  9.7667e-01,  5.2477e-01,
         -2.3003e+00, -7.2269e-01, -9.2569e-01, -1.7567e+00, -3.6968e-01,
          1.8250e-01, -4.2700e+00],
        [ 7.1626e-01, -1.0103e+00, -7.7511e-01,  1.4605e-01,  1.3433e+00,
         -1.4003e+00, -9.7766e-01, -7.8168e-01,  9.0255e-01, -1.7972e+00,
         -6.2352e-01,  2.2817e+00,  8.4076e-01,  2.0876e+00, -2.1013e-01,
         -1.3893e+00, -2.8570e+00,  1.7755e+00,  9.5380e-01,  5.5862e-01,
         -1.5729e+00,  3.3323e-01, -1.8909e+00, -9.2304e-01,  1.1712e+00,
         -6.0678e-01,  2.4857e+00],
        [ 4.3461e+00,  1.1707e+00,  3.2197e-01, -5.4095e-01,  3.1907e+00,
         -1.6958e+00, -1.5338e+00, -1.3403e+00,  3.6210e+00, -1.1337e+00,
         -1.6013e+00, -1.33

In [15]:
counts = logits_.exp()
negative_log_probs = -(counts / counts.sum(1, keepdim=True)).log() # -log_softmax
print(negative_log_probs)
avg_nll = negative_log_probs[range(len(y_train)), y_train].mean()
print(f"\navg_nll: {avg_nll.item():.4f}")

tensor([[1.9830, 3.2001, 3.0340,  ..., 4.0930, 3.5409, 7.9934],
        [3.4041, 5.1306, 4.8954,  ..., 2.9492, 4.7271, 1.6347],
        [0.9430, 4.1184, 4.9672,  ..., 3.1538, 6.2877, 2.5616],
        ...,
        [1.5182, 5.9423, 4.4533,  ..., 6.0189, 4.8528, 1.5838],
        [1.0300, 6.0669, 5.9431,  ..., 2.8214, 4.1422, 2.7339],
        [1.9531, 5.1556, 4.3018,  ..., 3.3831, 4.1081, 1.4716]],
       grad_fn=<NegBackward0>)

avg_nll: 2.4577


In [16]:
loss = -F.log_softmax(logits_, dim=1)
print(loss)
loss = loss[torch.arange(len(y_train)), y_train]
loss = loss.mean()
print(f'\n{loss.item():.4f}')

tensor([[1.9830, 3.2001, 3.0340,  ..., 4.0930, 3.5409, 7.9934],
        [3.4041, 5.1306, 4.8954,  ..., 2.9492, 4.7271, 1.6347],
        [0.9430, 4.1184, 4.9672,  ..., 3.1538, 6.2877, 2.5616],
        ...,
        [1.5182, 5.9423, 4.4533,  ..., 6.0189, 4.8528, 1.5838],
        [1.0300, 6.0669, 5.9431,  ..., 2.8214, 4.1422, 2.7339],
        [1.9531, 5.1556, 4.3018,  ..., 3.3831, 4.1081, 1.4716]],
       grad_fn=<NegBackward0>)

2.4577
